# Tutorial 1: Transformers from Scratch

**Prerequisites:** T00 (What Is Mechinterp?), basic Python/NumPy

**What you'll learn:**
- How a transformer processes text, step by step
- What each component does and why it exists
- The shapes and dimensions of every tensor
- How to think about transformers as mechinterp practitioners

---

## The Big Picture

A transformer converts a sequence of tokens into a sequence of predictions. The architecture has a beautifully simple structure:

```
tokens → [Embed + PosEmbed] → residual stream
                                    ↓
                              [Layer 0: Attention → MLP]
                                    ↓
                              [Layer 1: Attention → MLP]
                                    ↓
                                  ...
                                    ↓
                              [LayerNorm → Unembed] → logits → predictions
```

The key insight for mechinterp: **every component reads from and writes to a shared "residual stream."** This stream is a vector (of dimension `d_model`) at each token position, and it accumulates contributions from every layer.

Let's build this up piece by piece.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

# Create a small model we can fully inspect
cfg = HookedTransformerConfig(
    n_layers=2,     # Just 2 layers so we can trace everything
    d_model=32,     # 32-dimensional residual stream
    n_ctx=64,       # Max 64 tokens
    d_head=8,       # Each attention head works in 8 dimensions
    n_heads=4,      # 4 attention heads per layer (4 * 8 = 32 = d_model)
    d_vocab=100,    # 100-token vocabulary
)
model = HookedTransformer(cfg)

# Initialize with random weights so activations are non-trivial
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([1, 42, 17, 88, 55])  # 5 tokens
print(f'Model: {cfg.n_layers} layers, {cfg.n_heads} heads, d_model={cfg.d_model}')
print(f'Input: {len(tokens)} tokens')

## Step 1: Token Embedding

Each token ID is converted to a vector using a lookup table `W_E` (the embedding matrix).

- **Input**: token IDs, shape `[seq_len]`
- **W_E shape**: `[d_vocab, d_model]` — one row per token in the vocabulary
- **Output**: `[seq_len, d_model]` — each token is now a 32-dimensional vector

In [ ]:
# The embedding matrix
W_E = model.embed.W_E
print(f'W_E shape: {W_E.shape}  (one row per vocab token)')

# Embedding is just a lookup: W_E[token_id]
token_embeddings = W_E[np.array(tokens)]  # Convert to numpy for indexing
print(f'Token embeddings shape: {token_embeddings.shape}')
print(f'Embedding for token 42: {np.array(token_embeddings[1, :8]).round(3)}...')

## Step 2: Positional Embedding

Attention is permutation-equivariant — it doesn't inherently know token order. Positional embeddings tell the model *where* each token is.

- **W_pos shape**: `[n_ctx, d_model]` — one vector per possible position
- The residual stream starts as: `residual = W_E[tokens] + W_pos[:seq_len]`

In [ ]:
W_pos = model.pos_embed.W_pos
print(f'W_pos shape: {W_pos.shape}  (one row per position)')

pos_embeddings = W_pos[:len(tokens)]
print(f'Position embeddings for our 5 tokens: {pos_embeddings.shape}')

# The residual stream starts here
residual_start = token_embeddings + pos_embeddings
print(f'\nInitial residual stream: {residual_start.shape}')
print(f'This is what flows into Layer 0.')

## Step 3: Attention (The Heart of the Transformer)

Each attention layer has multiple **heads**. Each head independently:
1. **Queries**: "What am I looking for?" — projects residual stream through `W_Q`
2. **Keys**: "What do I contain?" — projects through `W_K`
3. **Values**: "What information do I carry?" — projects through `W_V`
4. **Attention pattern**: Query-key dot product → softmax → which positions to attend to
5. **Output**: Weighted sum of values → projected through `W_O` back to residual stream

### Key shapes (IRTK convention):
- `W_Q, W_K, W_V`: `[n_heads, d_model, d_head]` — per-head projection
- `W_O`: `[n_heads, d_head, d_model]` — per-head output projection

In [ ]:
# Let's trace attention in layer 0
attn = model.blocks[0].attn

print('Attention weight shapes:')
print(f'  W_Q: {attn.W_Q.shape}  [n_heads, d_model, d_head]')
print(f'  W_K: {attn.W_K.shape}  [n_heads, d_model, d_head]')
print(f'  W_V: {attn.W_V.shape}  [n_heads, d_model, d_head]')
print(f'  W_O: {attn.W_O.shape}  [n_heads, d_head, d_model]')
print()
print(f'Each head projects from d_model={cfg.d_model} down to d_head={cfg.d_head},')
print(f'computes attention in that small space, then projects back up.')

In [ ]:
# Run the model and look at the attention patterns
logits, cache = model.run_with_cache(tokens)

# The attention pattern tells us: for each head, for each query position,
# how much does it attend to each key position?
patterns = cache['blocks.0.attn.hook_pattern']  # [n_heads, seq, seq]
print(f'Attention patterns shape: {patterns.shape}')
print(f'  = [{cfg.n_heads} heads, {len(tokens)} queries, {len(tokens)} keys]')
print()

# Each row sums to 1 (softmax)
print('Head 0 pattern (rows=queries, cols=keys):')
p = np.array(patterns[0]).round(2)
print(p)
print(f'Row sums: {p.sum(axis=1).round(2)}  (all ~1.0 due to softmax)')
print()
print('Notice: the pattern is lower-triangular (causal masking).')
print('Token 0 can only attend to itself. Token 4 can attend to all 5.')

## Step 4: MLP (Multi-Layer Perceptron)

After attention moves information between positions, the MLP processes each position independently:

```
MLP(x) = activation_fn(x @ W_in) @ W_out
```

- `W_in`: `[d_model, d_mlp]` — expands to a larger intermediate space
- `W_out`: `[d_mlp, d_model]` — projects back to residual stream
- `d_mlp` is typically `4 * d_model`

MLPs are where **knowledge is stored** — factual associations, patterns, and features.

In [ ]:
mlp = model.blocks[0].mlp
print(f'MLP weight shapes:')
print(f'  W_in:  {mlp.W_in.shape}   [d_model, d_mlp]  (expand)')
print(f'  W_out: {mlp.W_out.shape}  [d_mlp, d_model]  (compress back)')
print()
print(f'd_mlp = {mlp.W_in.shape[1]} = {mlp.W_in.shape[1] // cfg.d_model}x d_model')
print()

# Look at pre- and post-activation
pre_act = cache['blocks.0.mlp.hook_pre']   # [seq, d_mlp]
post_act = cache['blocks.0.mlp.hook_post']  # [seq, d_mlp]
print(f'Pre-activation shape: {pre_act.shape}  (before nonlinearity)')
print(f'Post-activation shape: {post_act.shape}  (after nonlinearity)')
print(f'\nThe activation function kills some neurons (sets them to ~0).')
print(f'Active neurons at position 0: {int(jnp.sum(jnp.abs(post_act[0]) > 0.01))} '
      f'out of {post_act.shape[1]}')

## Step 5: The Residual Stream

This is the most important concept for mechinterp. Each layer's attention and MLP **add** their output to the residual stream:

```
residual_0 = embed + pos_embed
residual_0 += Attention_0(residual_0)    # attention reads and writes
residual_0 += MLP_0(residual_0)          # MLP reads and writes
residual_1 = residual_0                  # passed to next layer
residual_1 += Attention_1(residual_1)
residual_1 += MLP_1(residual_1)
output = Unembed(LayerNorm(residual_1))  # final prediction
```

The residual stream is a **sum of all contributions**. This means we can decompose it!

In [ ]:
# Track the residual stream through the model
print('Residual stream norms at each stage (last position):')
print(f'  After embed:  {float(jnp.linalg.norm(cache["blocks.0.hook_resid_pre"][-1])):.4f}')

for l in range(cfg.n_layers):
    # After attention
    attn_out = cache[f'blocks.{l}.hook_attn_out'][-1]
    print(f'  Layer {l} attn contribution: {float(jnp.linalg.norm(attn_out)):.4f}')
    
    # After MLP
    mlp_out = cache[f'blocks.{l}.hook_mlp_out'][-1]
    print(f'  Layer {l} MLP contribution:  {float(jnp.linalg.norm(mlp_out)):.4f}')
    
    # Full residual
    resid = cache[f'blocks.{l}.hook_resid_post'][-1]
    print(f'  Residual after layer {l}:   {float(jnp.linalg.norm(resid)):.4f}')
    print()

## Step 6: Unembedding (Making Predictions)

The final residual stream is converted to predictions over the vocabulary:

```
logits = LayerNorm(residual_final) @ W_U + b_U
```

- `W_U`: `[d_model, d_vocab]` — one column per vocab token
- The logit for token `t` is the dot product of the residual with column `t` of `W_U`
- Higher logit = model thinks that token is more likely to come next

In [ ]:
W_U = model.unembed.W_U
print(f'W_U shape: {W_U.shape}  [d_model, d_vocab]')
print()

# The model's prediction at the last position
last_logits = np.array(logits[-1])  # [d_vocab]
top_5 = np.argsort(last_logits)[::-1][:5]
print('Top 5 predictions at the last position:')
for rank, tok in enumerate(top_5):
    print(f'  #{rank+1}: token {tok} (logit = {last_logits[tok]:.4f})')
print()
print('With random weights, these predictions are meaningless.')
print('With trained weights (e.g. GPT-2), they produce coherent text!')

## Summary: The Transformer in One Picture

```
Token IDs:  [1, 42, 17, 88, 55]
                    ↓
          W_E lookup + W_pos
                    ↓
Residual: [seq=5, d_model=32]  ← this is the central object
                    ↓
        ┌─── Layer 0 ───┐
        │  LayerNorm     │
        │  4 Attn Heads  │ ← move info between positions
        │  + residual    │
        │  LayerNorm     │
        │  MLP           │ ← transform info at each position
        │  + residual    │
        └────────────────┘
                    ↓
        ┌─── Layer 1 ───┐
        │  (same)        │
        └────────────────┘
                    ↓
           LayerNorm → W_U
                    ↓
Logits:   [seq=5, d_vocab=100]  ← prediction for each position
```

### Key Dimensions to Remember

| Symbol | Meaning | Typical Size | Our Model |
|--------|---------|-------------|----------|
| `d_model` | Residual stream width | 768 (GPT-2) | 32 |
| `n_heads` | Attention heads per layer | 12 (GPT-2) | 4 |
| `d_head` | Per-head dimension | 64 (GPT-2) | 8 |
| `d_mlp` | MLP intermediate size | 3072 (GPT-2) | 128 |
| `d_vocab` | Vocabulary size | 50257 (GPT-2) | 100 |
| `n_ctx` | Max context length | 1024 (GPT-2) | 64 |

**Next: [T02 — Hooks and Caching](T02_hooks_and_caching.ipynb)** — Learn how to use IRTK's hook system to inspect and modify any internal computation.